# LIDC Prototype Sanity-Check Training Notebook

**Goal**: Verify that all .npz cached samples are learnable for full training.

**What this notebook does:**
1. Discovers and inspects all .npz cache files
2. Auto-detects image/mask keys and patient IDs
3. Builds a PyTorch dataset from all samples
4. Reports data health statistics
5. Trains a SegResNet model on 80/20 train/val split
6. Plots loss/dice curves and visualizes predictions

**Status**: Sanity check — trains on full dataset.


In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted")

In [ ]:
# Step 2: Install MONAI (Colab has numpy/torch/scipy already)
!pip install -q monai
print("✓ MONAI installed")

In [ ]:
import os
import glob
import json
import re
import random
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from monai.networks.nets import SegResNet
from monai.losses import DiceLoss

from tqdm import tqdm

warnings.filterwarnings('ignore')

In [ ]:
# Validate environment
print(f"\n✓ Environment:")
print(f"  Python: {torch.__version__[:3]} torch")
print(f"  NumPy: {np.__version__}")
print(f"  Device: {('GPU' if torch.cuda.is_available() else 'CPU')}")

## Step 1: Configuration

In [ ]:
# Configuration
CONFIG = {
    'root': '/content/drive/MyDrive/LIDC_PROTOTYPE_25D',
    'seed': 42,
    'image_size': (192, 192),
    # Dataset
    'batch_size': 8,
    'num_workers': 0,  # Colab compatibility
    'pin_memory': True,
    # Training
    'num_epochs': 100,
    'learning_rate': 1e-3,
    'loss_name': 'dicece',  # 'dicece' or 'dice'
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
}

np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed(CONFIG['seed'])

print("Configuration:")
for k, v in CONFIG.items():
    if k != 'device':
        print(f"  {k}: {v}")

## Step 2: Discover and Inspect Cache Files

In [ ]:
# Find all .npz files
root = CONFIG['root']

npz_files = sorted(glob.glob(os.path.join(root, '**/*.npz'), recursive=True))

print(f"\n{'='*100}")
print(f"DATA DISCOVERY (ALL .npz FILES)")
print(f"{'='*100}\n")
print(f"Cache root: {root}")
print(f"Total .npz files found: {len(npz_files)}\n")
assert len(npz_files) > 0, "ERROR: No .npz files found! Check cache root."
print("Sample files:")
for f in npz_files[:5]:
    print(f"  {os.path.relpath(f, root)}")
if len(npz_files) > 5:
    print(f"  ... and {len(npz_files) - 5} more")

In [ ]:
# Inspect first few .npz files
print(f"\n{'='*100}")
print(f"NPZ FILE INSPECTION")
print(f"{'='*100}\n")

for npz_path in npz_files[:3]:
    print(f"File: {os.path.basename(npz_path)}")
    try:
        data = np.load(npz_path, allow_pickle=True)
        print(f"  Keys: {list(data.keys())}")
        for key in data.keys():
            arr = data[key]
            if isinstance(arr, np.ndarray):
                print(f"    {key}: shape={arr.shape}, dtype={arr.dtype}")
            else:
                print(f"    {key}: type={type(arr).__name__}")
    except Exception as e:
        print(f"  ERROR: {e}")
    print()

## Step 3: Auto-Detect Keys and Extract Patient IDs

In [ ]:
def auto_detect_keys(npz_data_dict):
    """
    Detect image and mask keys from npz file.
    Returns: (image_key, mask_key)
    """
    image_candidates = ['images', 'image', 'input', 'x', 'img', 'patch', 'volume', 'data']
    mask_candidates = ['masks', 'mask', 'target', 'y', 'label', 'labels', 'seg', 'segmentation']
    
    keys = list(npz_data_dict.keys())
    
    # Find image key
    image_key = None
    for cand in image_candidates:
        if cand in keys:
            image_key = cand
            break
    
    # Find mask key
    mask_key = None
    for cand in mask_candidates:
        if cand in keys:
            mask_key = cand
            break
    
    return image_key, mask_key

def extract_patient_id(npz_path):
    """
    Extract patient ID from file path.
    Tries to match LIDC-IDRI-XXXX pattern.
    """
    match = re.search(r'(LIDC-IDRI-\d{4})', npz_path)
    if match:
        return match.group(1)
    
    # Fallback: use directory name
    parent_dir = os.path.basename(os.path.dirname(npz_path))
    if 'LIDC' in parent_dir:
        return parent_dir
    
    return os.path.splitext(os.path.basename(npz_path))[0]

print("✓ Auto-detection functions defined")

## Step 4: Build Dataset Class

In [ ]:
class NPZSegmentationDataset(Dataset):
    """
    Load 2.5D segmentation samples from .npz cache files.
    Handles shape variations and normalizes automatically.
    """
    
    def __init__(self, npz_paths, image_key='images', mask_key='masks', 
                 patient_ids=None, transform=None):
        self.npz_paths = npz_paths
        self.image_key = image_key
        self.mask_key = mask_key
        self.patient_ids = patient_ids or [None] * len(npz_paths)
        self.transform = transform
        self.samples = []
        
        # Pre-load metadata
        for idx, npz_path in enumerate(self.npz_paths):
            try:
                data = np.load(npz_path, allow_pickle=True)
                img = data[self.image_key].astype(np.float32)
                mask = data[self.mask_key].astype(np.uint8)
                
                # Handle shape variations
                # Image: expect (N, 5, H, W) or (N, H, W, 5)
                if img.ndim == 4:
                    if img.shape[1] == 5:  # (N, 5, H, W)
                        pass
                    elif img.shape[-1] == 5:  # (N, H, W, 5) -> transpose
                        img = img.transpose(0, 3, 1, 2)
                    n_samples = img.shape[0]
                elif img.ndim == 3:
                    # Single sample (5, H, W) -> add batch dim
                    if img.shape[0] == 5:
                        img = img[np.newaxis]  # (1, 5, H, W)
                        n_samples = 1
                    else:
                        raise ValueError(f"Unexpected image shape: {img.shape}")
                else:
                    raise ValueError(f"Unexpected image dims: {img.ndim}")
                
                # Mask: expect (N, H, W), (N, 1, H, W), or (N, H, W, 1)
                if mask.ndim == 3:  # (N, H, W)
                    pass
                elif mask.ndim == 4:  # (N, 1, H, W) or (N, H, W, 1)
                    if mask.shape[1] == 1:
                        mask = mask.squeeze(1)  # (N, H, W)
                    elif mask.shape[-1] == 1:
                        mask = mask.squeeze(-1)  # (N, H, W)
                elif mask.ndim == 2:  # (H, W) -> add batch dim
                    mask = mask[np.newaxis]  # (1, H, W)
                    n_samples = 1
                else:
                    raise ValueError(f"Unexpected mask dims: {mask.ndim}")
                
                # Binarize mask
                mask = (mask > 0).astype(np.uint8)
                
                # Store samples
                for i in range(n_samples):
                    self.samples.append({
                        'image': img[i],
                        'mask': mask[i],
                        'patient_id': self.patient_ids[idx],
                        'npz_idx': idx,
                        'sample_idx': i
                    })
            except Exception as e:
                print(f"Warning: Failed to load {npz_path}: {e}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = sample['image'].copy()  # (5, H, W)
        mask = sample['mask'].copy()    # (H, W)
        
        # Normalize image per-channel
        for ch in range(image.shape[0]):
            ch_mean = image[ch].mean()
            ch_std = image[ch].std()
            if ch_std > 0:
                image[ch] = (image[ch] - ch_mean) / (ch_std + 1e-6)
        
        # Convert to tensors
        image = torch.from_numpy(image).float()  # (5, H, W)
        mask = torch.from_numpy(mask).float()    # (H, W)
        mask = mask.unsqueeze(0)  # (1, H, W) for compatibility
        
        if self.transform:
            image, mask = self.transform(image, mask)
        
        return {
            'image': image,
            'mask': mask,
            'patient_id': sample['patient_id']
        }

print("✓ Dataset class defined")

## Step 5: Data Health Check

In [ ]:
# Build file metadata
file_metadata = []
for npz_path in npz_files:
    try:
        data = np.load(npz_path, allow_pickle=True)
        img_key, mask_key = auto_detect_keys(data)
        pid = extract_patient_id(npz_path)
        file_metadata.append({
            'path': npz_path,
            'patient_id': pid,
            'img_key': img_key,
            'mask_key': mask_key,
            'usable': img_key is not None and mask_key is not None
        })
    except Exception as e:
        print(f"Warning: {os.path.basename(npz_path)}: {e}")

usable_files = [f for f in file_metadata if f['usable']]
unique_patients = len(set(f['patient_id'] for f in usable_files))

print(f"\n{'='*100}")
print(f"DATA HEALTH CHECK")
print(f"{'='*100}\n")
print(f"Total .npz files: {len(npz_files)}")
print(f"Usable files: {len(usable_files)}")
print(f"Unique patients: {unique_patients}")
print(f"Success rate: {100 * len(usable_files) / len(npz_files):.1f}%\n")

# Image key consensus
img_keys = [f['img_key'] for f in usable_files]
mask_keys = [f['mask_key'] for f in usable_files]
print(f"Image key used: {max(set(img_keys), key=img_keys.count)}")
print(f"Mask key used: {max(set(mask_keys), key=mask_keys.count)}")

In [ ]:
# Use the most common keys
IMAGE_KEY = max(set(img_keys), key=img_keys.count)
MASK_KEY = max(set(mask_keys), key=mask_keys.count)

# Build dataset from usable files
usable_paths = [f['path'] for f in usable_files]
usable_pids = [f['patient_id'] for f in usable_files]

print(f"\nBuilding dataset with:")
print(f"  Image key: {IMAGE_KEY}")
print(f"  Mask key: {MASK_KEY}")
print(f"  Files: {len(usable_paths)}")

full_dataset = NPZSegmentationDataset(
    usable_paths, 
    image_key=IMAGE_KEY, 
    mask_key=MASK_KEY,
    patient_ids=usable_pids
)

print(f"\n✓ Dataset created with {len(full_dataset)} total samples")
print(f"  Samples per file: {len(full_dataset) / len(usable_paths):.1f}")

In [ ]:
# Detailed statistics
print(f"\nDataset statistics:")

empty_mask_count = 0
positive_pixels = []
image_shapes = []

for sample in full_dataset.samples:
    mask = sample['mask']
    if mask.max() == 0:
        empty_mask_count += 1
    else:
        positive_pixels.append(mask.sum())
    image_shapes.append(sample['image'].shape)

print(f"  Total samples: {len(full_dataset)}")
print(f"  Empty masks: {empty_mask_count} ({100 * empty_mask_count / len(full_dataset):.1f}%)")
if positive_pixels:
    print(f"  Positive pixels (mean): {np.mean(positive_pixels):.0f}")
    print(f"  Positive pixels (std): {np.std(positive_pixels):.0f}")
    print(f"  Positive pixels (range): [{np.min(positive_pixels):.0f}, {np.max(positive_pixels):.0f}]")

unique_shapes = set(image_shapes)
if len(unique_shapes) == 1:
    print(f"  Image shape (consistent): {list(unique_shapes)[0]}")
else:
    print(f"  Image shapes (INCONSISTENT): {unique_shapes}")

# Define unique patient IDs for later use
unique_pids = sorted(set(usable_pids))
print(f"\nUnique patients: {len(unique_pids)}")

In [ ]:
## Step 6: Define Diagnostic Helper Functions

def analyze_split_balance(dataset, train_indices, val_indices):
    """Analyze patient and sample distribution across train/val splits."""
    train_samples = [dataset.samples[i] for i in train_indices]
    val_samples = [dataset.samples[i] for i in val_indices]
    
    train_patients = set(s['patient_id'] for s in train_samples)
    val_patients = set(s['patient_id'] for s in val_samples)
    
    train_samples_per_patient = {}
    val_samples_per_patient = {}
    
    for s in train_samples:
        pid = s['patient_id']
        train_samples_per_patient[pid] = train_samples_per_patient.get(pid, 0) + 1
    
    for s in val_samples:
        pid = s['patient_id']
        val_samples_per_patient[pid] = val_samples_per_patient.get(pid, 0) + 1
    
    return {
        'train_num_patients': len(train_patients),
        'val_num_patients': len(val_patients),
        'train_num_samples': len(train_samples),
        'val_num_samples': len(val_samples),
        'train_samples_per_patient': train_samples_per_patient,
        'val_samples_per_patient': val_samples_per_patient,
    }

def summarize_mask_foreground(dataset, train_indices=None, val_indices=None):
    """Analyze foreground pixel statistics for masks."""
    def compute_fg_stats(sample_list):
        fg_pixels = []
        empty_count = 0
        
        for sample in sample_list:
            mask = sample['mask']
            fg = mask.sum()
            if fg > 0:
                fg_pixels.append(float(fg))
            else:
                empty_count += 1
        
        if fg_pixels:
            stats = {
                'count': len(fg_pixels),
                'empty': empty_count,
                'mean': np.mean(fg_pixels),
                'median': np.median(fg_pixels),
                'min': np.min(fg_pixels),
                'max': np.max(fg_pixels),
                'std': np.std(fg_pixels),
            }
        else:
            stats = {'count': 0, 'empty': len(sample_list), 'mean': 0, 'median': 0}
        
        return stats
    
    result = {}
    if train_indices is not None and val_indices is not None:
        train_samples = [dataset.samples[i] for i in train_indices]
        val_samples = [dataset.samples[i] for i in val_indices]
        result['train'] = compute_fg_stats(train_samples)
        result['val'] = compute_fg_stats(val_samples)
    else:
        all_samples = dataset.samples
        result['all'] = compute_fg_stats(all_samples)
    
    return result

def get_loss_function(loss_name="dicece"):
    """Create loss function based on name."""
    if loss_name.lower() == "dicece":
        from monai.losses import DiceCELoss
        return DiceCELoss(sigmoid=True)
    else:
        return DiceLoss(sigmoid=True) + nn.BCEWithLogitsLoss()

def evaluate_threshold_sweep(model, dataloader, device, thresholds=[0.3, 0.4, 0.5]):
    """Compute Dice at multiple thresholds on validation set."""
    model.eval()
    threshold_dices = {t: [] for t in thresholds}
    
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            logits = model(images)
            probs = torch.sigmoid(logits)
            
            for threshold in thresholds:
                pred = (probs > threshold).float()
                intersection = (pred * masks).sum()
                union = pred.sum() + masks.sum()
                dice = 2.0 * intersection / (union + 1e-6)
                threshold_dices[threshold].append(dice.item())
    
    return {t: np.mean(dices) for t, dices in threshold_dices.items()}

def visualize_val_predictions(model, val_loader, device, num_samples=8):
    """Visualize validation predictions: center slice, GT mask, probabilities, threshold pred."""
    model.eval()
    
    collected_images = []
    collected_masks = []
    collected_preds = []
    
    with torch.no_grad():
        for batch in val_loader:
            if len(collected_images) >= num_samples:
                break
            
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            logits = model(images)
            preds = torch.sigmoid(logits)
            
            collected_images.extend(images.cpu())
            collected_masks.extend(masks.cpu())
            collected_preds.extend(preds.cpu())
    
    num_to_show = min(num_samples, len(collected_images))
    
    fig, axes = plt.subplots(4, num_to_show, figsize=(3 * num_to_show, 12))
    if num_to_show == 1:
        axes = axes.reshape(-1, 1)
    
    fig.suptitle("Validation Predictions", fontsize=14, fontweight='bold')
    
    for i in range(num_to_show):
        img = collected_images[i]
        mask_gt = collected_masks[i][0]
        pred_prob = collected_preds[i][0]
        pred_binary = pred_prob > 0.5
        
        # Row 0: Center input slice
        ax = axes[0, i]
        center_slice = img[2].numpy()
        ax.imshow(center_slice, cmap='gray')
        ax.set_title(f"Input {i}", fontsize=10)
        ax.axis('off')
        
        # Row 1: Ground truth mask
        ax = axes[1, i]
        ax.imshow(center_slice, cmap='gray')
        ax.contour(mask_gt.numpy(), levels=[0.5], colors='red', linewidths=2)
        ax.set_title(f"GT {i}", fontsize=10)
        ax.axis('off')
        
        # Row 2: Predicted probability map
        ax = axes[2, i]
        ax.imshow(pred_prob.numpy(), cmap='hot', vmin=0, vmax=1)
        ax.set_title(f"Prob {i}", fontsize=10)
        ax.axis('off')
        
        # Row 3: Thresholded prediction mask
        ax = axes[3, i]
        ax.imshow(center_slice, cmap='gray')
        ax.contour(pred_binary.numpy().astype(float), levels=[0.5], colors='lime', linewidths=2)
        ax.set_title(f"Pred {i}", fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('/tmp/val_predictions.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Visualized {num_to_show} validation predictions")

print("✓ Diagnostic helper functions defined")

In [ ]:
## Step 7: Dataset Diagnostics

print(f"\n{'='*100}")
print(f"DATASET DIAGNOSTICS")
print(f"{'='*100}\n")

# Full dataset statistics
fg_stats_full = summarize_mask_foreground(full_dataset)
print("Full dataset foreground statistics:")
for key, stats in fg_stats_full.items():
    print(f"  {key}:")
    print(f"    Samples with foreground: {stats['count']}")
    print(f"    Empty masks: {stats['empty']}")
    if stats['count'] > 0:
        print(f"    Mean FG pixels: {stats['mean']:.0f}")
        print(f"    Median FG pixels: {stats['median']:.0f}")
        print(f"    Range: [{stats['min']:.0f}, {stats['max']:.0f}]")
        print(f"    Std: {stats['std']:.0f}")

# Samples per patient
print(f"\nSamples per patient (full dataset):")
samples_per_patient = {}
for pid in unique_pids:
    count = sum(1 for s in full_dataset.samples if s['patient_id'] == pid)
    samples_per_patient[pid] = count

for pid in sorted(samples_per_patient.keys()):
    count = samples_per_patient[pid]
    print(f"  {pid}: {count} samples")

print(f"  Mean: {np.mean(list(samples_per_patient.values())):.1f}")
print(f"  Std: {np.std(list(samples_per_patient.values())):.1f}")

## Step 8: Visualize Sample

In [ ]:
# Show a sample
sample_idx = 0
sample = full_dataset[sample_idx]
image = sample['image'].numpy()  # (5, 192, 192)
mask = sample['mask'].numpy()[0]  # (192, 192)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle(f"Sample {sample_idx}: {sample['patient_id']}", fontsize=14, fontweight='bold')

# Show each of 5 channels
for ch in range(5):
    ax = axes[ch // 3, ch % 3]
    ax.imshow(image[ch], cmap='gray')
    ax.set_title(f"Channel {ch - 2:+d}")
    ax.axis('off')

# Bottom right: center + mask
ax = axes[1, 2]
ax.imshow(image[2], cmap='gray')
ax.contour(mask, levels=[0.5], colors='lime', linewidths=2)
ax.set_title(f"Center + Mask")
ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/sample_viz.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Sample visualization saved")

## Step 9: Create Train/Val Splits

In [ ]:
# Unique patient IDs
unique_pids = sorted(set(s['patient_id'] for s in full_dataset.samples))

# Patient-level split (FULL CACHE - all patients!)
random.seed(CONFIG['seed'])
shuffled_pids = unique_pids.copy()
random.shuffle(shuffled_pids)

split_idx = int(0.8 * len(shuffled_pids))
train_pids = set(shuffled_pids[:split_idx])
val_pids = set(shuffled_pids[split_idx:])

train_idx = [i for i, s in enumerate(full_dataset.samples) if s['patient_id'] in train_pids]
val_idx = [i for i, s in enumerate(full_dataset.samples) if s['patient_id'] in val_pids]

# Create train/val subsets
train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)

print("=" * 80)
print("DATA SPLIT SUMMARY (FULL CACHE)")
print("=" * 80)
print(f"Unique patients   : {len(unique_pids)}")
print(f"Train patients    : {len(train_pids)}")
print(f"Val patients      : {len(val_pids)}")
print(f"Train samples     : {len(train_dataset)}")
print(f"Val samples       : {len(val_dataset)}")
print("=" * 80)

## Step 10: Create DataLoaders

In [ ]:
# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['pin_memory']
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['pin_memory']
)

print(f"DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

# Test a batch
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  Image: {batch['image'].shape}")
print(f"  Mask: {batch['mask'].shape}")

## Step 11: Define Model & Loss

In [ ]:
# Lightweight SegResNet for 2D segmentation
model = SegResNet(
    spatial_dims=2,
    in_channels=5,
    out_channels=1,
    init_filters=8,  # Small for Colab
    blocks_down=(1, 2, 2, 4),
    blocks_up=(1, 1, 1),
).to(CONFIG['device'])

print(f"Model created: {model.__class__.__name__}")
print(f"  Device: {CONFIG['device']}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Loss function (now configurable)
combined_loss = get_loss_function(CONFIG['loss_name'])

print(f"Loss function: {CONFIG['loss_name']}")
print(f"Optimizer: Adam lr={CONFIG['learning_rate']}")

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])

## Step 12: Training Loop Functions

In [ ]:
def dice_score(logits, target):
    """Compute Dice from logits."""
    probs = torch.sigmoid(logits)
    pred = (probs > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    return 2.0 * intersection / (union + 1e-6)

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    total_dice = 0.0
    num_batches = 0
    
    for batch in dataloader:
        images = batch['image'].to(device)  # (B, 5, H, W)
        masks = batch['mask'].to(device)    # (B, 1, H, W)
        
        optimizer.zero_grad()
        logits = model(images)  # (B, 1, H, W)
        
        loss = criterion(logits, masks)
        dice = dice_score(logits, masks)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_dice += dice.item()
        num_batches += 1
    
    return total_loss / num_batches, total_dice / num_batches

def eval_one_epoch(model, dataloader, criterion, device):
    """Evaluate for one epoch."""
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            logits = model(images)
            loss = criterion(logits, masks)
            dice = dice_score(logits, masks)
            
            total_loss += loss.item()
            total_dice += dice.item()
            num_batches += 1
    
    return total_loss / num_batches, total_dice / num_batches

print("✓ Training functions defined")

## Step 13: Full Cache Training

## Step 13B: Main Training Loop

In [ ]:
print(f"\n{'='*100}")
print(f"FULL CACHE TRAINING: {len(train_pids)} train + {len(val_pids)} val patients")
print(f"{'='*100}\n")

print(f"Train: {len(train_dataset)} samples from {len(train_pids)} patients")
print(f"Val: {len(val_dataset)} samples from {len(val_pids)} patients")
print(f"Epochs: {CONFIG['num_epochs']}")
print(f"Batch size: {CONFIG['batch_size']}\n")

# Reset model
model = SegResNet(
    spatial_dims=2,
    in_channels=5,
    out_channels=1,
    init_filters=8,
    blocks_down=(1, 2, 2, 4),
    blocks_up=(1, 1, 1),
).to(CONFIG['device'])

optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])

# Track metrics
train_losses = []
train_dices = []
val_losses = []
val_dices = []
best_val_dice = 0.0
best_model_state = None

# Train
for epoch in range(1, CONFIG['num_epochs'] + 1):
    train_loss, train_dice = train_one_epoch(
        model, train_loader, combined_loss, optimizer, CONFIG['device']
    )
    val_loss, val_dice = eval_one_epoch(
        model, val_loader, combined_loss, CONFIG['device']
    )
    
    train_losses.append(train_loss)
    train_dices.append(train_dice)
    val_losses.append(val_loss)
    val_dices.append(val_dice)
    
    # Track best model
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_model_state = model.state_dict().copy()
    
    print(f"Epoch {epoch}: "
          f"TrLoss={train_loss:.4f} TrDice={train_dice:.4f} | "
          f"VaLoss={val_loss:.4f} VaDice={val_dice:.4f}")

print(f"\n✓ Training complete")
print(f"Best Val Dice: {best_val_dice:.4f}")

In [ ]:
# Plot train/val curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, marker='o', label='Train')
ax1.plot(val_losses, marker='s', label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Full Cache Training: Loss')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(train_dices, marker='o', label='Train')
ax2.plot(val_dices, marker='s', label='Val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice Score')
ax2.set_title('Full Cache Training: Dice')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig('/tmp/mini_train_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Train Dice:")
print(f"  Initial: {train_dices[0]:.4f}")
print(f"  Final: {train_dices[-1]:.4f}")
print(f"  Improvement: {train_dices[-1] - train_dices[0]:.4f}")

In [ ]:
## Step 15: Threshold Sweep on Validation

# Reload best model
if best_model_state:
    best_model = SegResNet(
        spatial_dims=2,
        in_channels=5,
        out_channels=1,
        init_filters=8,
        blocks_down=(1, 2, 2, 4),
        blocks_up=(1, 1, 1),
    ).to(CONFIG['device'])
    best_model.load_state_dict(best_model_state)
else:
    best_model = model

# Sweep different thresholds
thresholds = [0.3, 0.4, 0.5]
threshold_dices = evaluate_threshold_sweep(best_model, val_loader, CONFIG['device'], thresholds)

print(f"\nValidation Dice at different thresholds:")
best_threshold = max(threshold_dices, key=threshold_dices.get)
for t in sorted(threshold_dices.keys()):
    marker = " ← BEST" if t == best_threshold else ""
    print(f"  Threshold {t:.1f}: {threshold_dices[t]:.4f}{marker}")

print(f"\nBest threshold: {best_threshold} (Dice={threshold_dices[best_threshold]:.4f})")

In [ ]:
## Step 16: Visualize Validation Predictions

# Use best model for visualization
visualize_val_predictions(best_model, val_loader, CONFIG['device'], num_samples=8)

In [ ]:
# Load best model and visualize
if best_model_state:
    model.load_state_dict(best_model_state)
model.eval()

with torch.no_grad():
    batch = next(iter(val_loader))
    images = batch['image'].to(CONFIG['device'])
    masks = batch['mask'].to(CONFIG['device'])
    logits = model(images)
    preds = torch.sigmoid(logits) > 0.5

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle("Best Model: Validation Predictions (Full Cache Training)", fontweight='bold')

for i in range(min(4, len(images))):
    img_center = images[i, 2].cpu().numpy()
    gt_mask = masks[i, 0].cpu().numpy()
    pred_mask = preds[i, 0].cpu().numpy()
    
    # Row 1: Image
    ax = axes[0, i]
    ax.imshow(img_center, cmap='gray')
    ax.set_title(f"Image {i}")
    ax.axis('off')
    
    # Row 2: Ground truth
    ax = axes[1, i]
    ax.imshow(img_center, cmap='gray')
    ax.contour(gt_mask, levels=[0.5], colors='red', linewidths=2)
    ax.set_title(f"GT {i}")
    ax.axis('off')
    
    # Row 3: Prediction
    ax = axes[2, i]
    ax.imshow(img_center, cmap='gray')
    ax.contour(pred_mask, levels=[0.5], colors='lime', linewidths=2)
    ax.set_title(f"Pred {i}")
    ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/mini_predictions.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Best model visualizations complete")

In [ ]:
# Final summary
print(f"\n{'='*100}")
print(f"TRAINING SUMMARY")
print(f"{'='*100}\n")

print(f"Dataset:")
print(f"  Total .npz files: {len(npz_files)}")
print(f"  Usable files: {len(usable_files)}")
print(f"  Unique patients: {len(unique_pids)}")
print(f"  Total samples: {len(full_dataset)}")

print(f"\nFull Cache Training Results:")
print(f"  Train patients: {len(train_pids)}")
print(f"  Val patients: {len(val_pids)}")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Val samples: {len(val_dataset)}")
print(f"  Best Val Dice: {best_val_dice:.4f}")
print(f"  ✓ Success" if best_val_dice > 0.3 else "  ⚠ Poor performance")

print(f"\nConclusion:")
if best_val_dice > 0.3:
    print(f"  ✅ FULL CACHE IS LEARNABLE - Training complete!")
else:
    print(f"  ⚠️  Check data balance or preprocessing quality")